In [ ]:
!pip install sentence-transformers
pip install streamlit

In [1]:
import pandas as pd
import re
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
from sentence_transformers import SentenceTransformer
import ast
import faiss
from sklearn.decomposition import TruncatedSVD
# import streamlit
# import json

In [ ]:
nltk.download('punkt_tab')
nltk.download('stopwords')

# Load to Memory & Merge Features and Desc.

In [2]:
games_df = pd.read_csv('games.csv')
metadata_df = pd.read_json('games_metadata.json', lines=True)

df = pd.merge(games_df, metadata_df, on='app_id', how='inner')

games_df = None
metadata_df = None

In [ ]:
# df.to_csv('Altered CSVs/merged_game_data.csv', index=False)

# MultiLabelBinarizer

In [ ]:
# df = pd.read_csv('Altered CSVs/merged_game_data.csv')

In [3]:
mlb = MultiLabelBinarizer()

# Convert the 'tags' column from a string representation of a list to an actual list
df['tags'] = df['tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

tag_matrix = mlb.fit_transform(df['tags'])

tags_df = pd.DataFrame(tag_matrix, columns=mlb.classes_)

df = pd.concat([df, tags_df], axis=1)

In [6]:
tags_df.head(5)

,1980s,1990's,2.5D,2D,2D Fighter,2D Platformer,360 Video,3D,3D Fighter,3D Platformer,...,Well-Written,Werewolves,Western,Wholesome,Word Game,World War I,World War II,Wrestling,Zombies,eSports
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# df.to_csv('Altered CSVs/MutliLabelBinarized.csv', index=False)

# Preprocessing

## Cleaning

In [ ]:
# df = pd.read_csv('Altered CSVs/MutliLabelBinarized.csv')

In [4]:
def clean_text(text):
    # Remove Special Characters
    text = re.sub(r'[^\w\s]', '', text)

    # Sets all characters to lowercase
    text = text.lower()

    # Removes URLs beginning with https, http, or www
    text = re.sub(r'https\S+|http\S+|www\S+', '', text, flags=re.MULTILINE)
    
    return text

In [5]:
df['description_filled'] = df['description']

In [6]:
df['description_filled'] = df['description_filled'].apply(clean_text)

In [7]:
rows = df['description_filled'].str.strip() == ''

df.loc[rows, 'description_filled'] = (
    df.loc[rows, 'title'] + ' ' +
    df.loc[rows, 'tags'].apply(' '.join)
)

df.loc[rows, 'description_filled'] = df['description_filled'].apply(clean_text)

In [8]:
# Print any rows with descriptions with only whitespace characters
empty_descriptions = df[df['description_filled'].str.strip() == '']
empty_descriptions

,app_id,title,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,...,Werewolves,Western,Wholesome,Word Game,World War I,World War II,Wrestling,Zombies,eSports,description_filled


In [20]:
# The app Descripition is literally null
# null_literal_descriptions = df[df['description'].str.strip() == 'null']
# null_literal_descriptions

In [37]:
# Print Fixed Descriptions
df.loc[rows, 'description_filled']

1                            brink agents of change action
7        men of war assault squad 2  deluxe edition upg...
18       borderlands 2 headhunter 4 wedding day massacr...
21       sniper elite 3  camouflage weapons pack advent...
30       the house in fata morgana original soundtrack ...
                               ...                        
50866                                   train sim world 4 
50867            i expect you to die 3 cog in the machine 
50868                                            payday 3 
50869                                          eternights 
50871                                           fatalzone 
Name: description_filled, Length: 10376, dtype: object

In [42]:
# Checking Random Description Columns
pd.set_option('display.max_colwidth', None)
print(df['description_filled'].sample(n=3))
pd.reset_option('display.max_colwidth')

45768                                             you are a journalist in search of your breakthrough story a promising lead takes you to the abandoned island of mythargia where you have to discover its buried history connecting two different dimensions
14326    go on adventures and solve puzzles on this feelgood journey with a brother and sister as they explore dreamscapes and befriend magical creatures lost in their imagination toto and gal must stick together and solve puzzles to find their way home
11449                                                                                                                                                                                a total war saga troy  ajax  diomedes simulation action strategy violent
Name: description_filled, dtype: object


In [ ]:
# df.to_csv('Altered CSVs/after_cleaning.csv', index=False)

## Tokenization & Lemmatization

In [ ]:
# df = pd.read_csv('Altered CSVs/after_cleaning.csv')

In [9]:
# Tokenize the descriptions
df['tokens'] = df['description_filled'].apply(word_tokenize)

# Remove Stop Words from Tokens
stop_words = set(stopwords.words('english'))

df['tokens'] = df['tokens'].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)

# Lemmatize the Tokens
lemmatizer = WordNetLemmatizer()
df['tokens'] = df['tokens'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)

In [10]:
# Remove Stop Words from Tokens
stop_words = set(stopwords.words('english'))

df['tokens'] = df['tokens'].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)

In [11]:
# Lemmatize the Tokens
lemmatizer = WordNetLemmatizer()
df['tokens'] = df['tokens'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)

In [120]:
# df.to_csv('Altered CSVs/after_tokenization.csv', index=False)

# Sentance Transformer / Embeddings



In [ ]:
# df = pd.read_csv('Altered CSVs/MutliLabelBinarized.csv')

In [12]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# df['title'] = df['title'].fillna('')
# df['description'] = df['description'].fillna('')

# Ensure tags (which are lists) are converted to a single string before concatenation
df['tags_text'] = df['tags'].apply(
    lambda x: ' '.join(x) if isinstance(x, (list, tuple)) else str(x)
)

texts = (df['title'] + ' ' + df['description_filled'] + ' ' + df['tags_text']).tolist()

text_embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

df['embedding'] = list(text_embeddings)

Batches:   0%|          | 0/1590 [00:00<?, ?it/s]

c:\Users\vince\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [122]:
# df.to_csv('Altered CSVs/with_embeddings.csv', index=False)

# Normalization

## Min-Max

In [ ]:
# df = pd.read_csv('Altered CSVs/with_embeddings.csv')

**Applied for Ratings**

In [13]:
rating_dict = {
    'Overwhelmingly Positive': 9,
    'Very Positive': 8,
    'Positive': 7,
    'Mostly Positive': 6,
    'Mixed': 5,
    'Mostly Negative': 4,
    'Negative': 3,
    'Very Negative': 2,
    'Overwhelmingly Negative': 1
}

df['rating_normalized'] = (df['rating'].map(rating_dict) - 1) / 8

## Log Normalizing

**Applied for Positive Ratio, Price, and Review Count**

In [14]:
df['positive_ratio_log'] = np.log1p(df['positive_ratio'])
df['price_log'] = np.log1p(df['price_final'])
df['user_reviews_log'] = np.log1p(df['user_reviews'])

In [125]:
# df.to_csv('Altered CSVs/normalized.csv', index=False)

# Concatenating Features

In [ ]:
# df = pd.read_csv('Altered CSVs/normalized.csv')

In [15]:
embeddings = text_embeddings

tags = tag_matrix

numerics = df[['price_log', 'positive_ratio_log', 'user_reviews_log']].values

X = np.hstack([numerics*0.6, embeddings*0.1, tags*2.0])

cs_matrix = cosine_similarity(X)

# np.save('hstack_data.npy', X)

np.save('cs_matrix.npy', cs_matrix)

# Cosine Similarity

In [ ]:
# df = pd.read_csv('Altered CSVs/merged_game_data.csv')

In [16]:
# df = pd.read_csv('Altered CSVs/merged_game_data.csv')
# df = pd.read_csv('Altered CSVs/normalized.csv')

# cs_matrix = np.load('cs_matrix.npy')

def recommend(df, appid, cs_matrix, k=10):
    
    index = df[df['app_id'] == appid].index[0]
    pos = df.index.get_loc(index)
    
    scores = cs_matrix[pos].copy()
    scores[pos] = -np.inf

    indexes = np.argsort(scores)[-k:][::-1]
    return df.iloc[indexes]

In [65]:
# Killing Floor 2
rec = recommend(df, 232090, cs_matrix)
rec[['app_id', 'title', 'rating', 'user_reviews', 'positive_ratio', 'price_final','tags', 'description']]

,app_id,title,rating,user_reviews,positive_ratio,price_final,tags,description
13179,412020,Metro Exodus,Very Positive,82022,89,30.0,[],
12687,323190,Frostpunk,Very Positive,77955,92,30.0,[],
14161,848450,Subnautica: Below Zero,Very Positive,74092,91,30.0,[],
48422,1167630,Teardown,Overwhelmingly Positive,71271,96,30.0,[],
13262,686810,Hell Let Loose,Very Positive,68869,84,29.0,[],
47947,601150,Devil May Cry 5,Overwhelmingly Positive,71034,95,30.0,[],
47405,8870,BioShock Infinite,Very Positive,99379,93,30.0,[],
15401,361420,ASTRONEER,Very Positive,94605,91,30.0,[],
47619,289650,Assassin's Creed® Unity,Mostly Positive,53273,76,30.0,[],
14401,359320,Elite Dangerous,Mostly Positive,68600,76,30.0,[],


In [74]:
# ELDEN RING
rec = recommend(df, 1245620, cs_matrix)
rec[['app_id', 'title', 'rating', 'user_reviews', 'positive_ratio', 'price_final','tags', 'description']]

,app_id,title,rating,user_reviews,positive_ratio,price_final,tags,description
13803,221100,DayZ,Mostly Positive,296845,74,45.0,[],
12689,1063730,New World,Mostly Positive,222345,70,40.0,[],
14163,1091500,Cyberpunk 2077,Very Positive,557051,80,60.0,[],
15719,1172620,Sea of Thieves 2023 Edition,Very Positive,253844,90,40.0,[],
14434,275850,No Man's Sky,Mostly Positive,209971,76,60.0,[],
48514,1238810,Battlefield™ V,Mostly Positive,141836,70,50.0,[],
14770,1086940,Baldur's Gate 3,Overwhelmingly Positive,269840,95,60.0,[],
15364,261550,Mount & Blade II: Bannerlord,Very Positive,177725,87,50.0,[],
13598,976730,Halo: The Master Chief Collection,Very Positive,192874,92,40.0,[],
12658,594650,Hunt: Showdown,Very Positive,136385,83,40.0,[],


In [75]:
# Titanfall 2
rec = recommend(df, 1237970, cs_matrix)
rec[['app_id', 'title', 'rating', 'user_reviews', 'positive_ratio', 'price_final','tags', 'description']]


,app_id,title,rating,user_reviews,positive_ratio,price_final,tags,description
47817,774171,Muse Dash,Very Positive,86591,90,3.0,[],
3372,322170,Geometry Dash,Very Positive,239081,93,4.0,[],
47548,238320,Outlast,Overwhelmingly Positive,75391,96,3.0,[],
15281,1794680,Vampire Survivors,Overwhelmingly Positive,197109,98,5.0,[],
47791,431960,Wallpaper Engine,Overwhelmingly Positive,637341,98,4.0,[],
50781,945360,Among Us,Very Positive,587821,92,3.0,[],
47394,3590,Plants vs. Zombies GOTY Edition,Overwhelmingly Positive,100864,97,5.0,[],
47790,433340,Slime Rancher,Overwhelmingly Positive,97997,97,5.0,[],
48013,674940,Stick Fight: The Game,Very Positive,89774,93,5.0,[],
47647,304430,INSIDE,Overwhelmingly Positive,45867,96,2.0,[],


In [76]:
# Insurgency Sandstorm
rec = recommend(df, 581320, cs_matrix)
rec[['app_id', 'title', 'rating', 'user_reviews', 'positive_ratio', 'price_final','tags', 'description']]

,app_id,title,rating,user_reviews,positive_ratio,price_final,tags,description
14166,220200,Kerbal Space Program,Overwhelmingly Positive,94712,95,10.0,[],
13504,22380,Fallout: New Vegas,Overwhelmingly Positive,147417,96,10.0,[],
47383,400,Portal,Overwhelmingly Positive,117868,98,10.0,[],
47523,219740,Don't Starve,Overwhelmingly Positive,87184,96,10.0,[],
47933,582660,Black Desert,Mostly Positive,49539,76,10.0,[],
47742,391540,Undertale,Overwhelmingly Positive,182534,96,10.0,[],
14774,1118200,People Playground,Overwhelmingly Positive,195164,98,10.0,[],
47832,470220,UNO,Mostly Positive,35495,73,10.0,[],
15278,960090,Bloons TD 6,Overwhelmingly Positive,239938,97,14.0,[],
50787,250900,The Binding of Isaac: Rebirth,Overwhelmingly Positive,225815,97,15.0,[],


In [77]:
# Halo The Master Chief Collection
rec = recommend(df, 976730, cs_matrix)
rec[['app_id', 'title', 'rating', 'user_reviews', 'positive_ratio', 'price_final','tags', 'description']]

,app_id,title,rating,user_reviews,positive_ratio,price_final,tags,description
15097,782330,DOOM Eternal,Very Positive,153466,91,40.0,[],
47760,394360,Hearts of Iron IV,Very Positive,176243,92,40.0,[],
15719,1172620,Sea of Thieves 2023 Edition,Very Positive,253844,90,40.0,[],
12658,594650,Hunt: Showdown,Very Positive,136385,83,40.0,[],
15397,489830,The Elder Scrolls V: Skyrim Special Edition,Very Positive,138503,94,40.0,[],
15809,534380,Dying Light 2 Stay Human,Mostly Positive,113138,79,30.0,[],
12740,294100,RimWorld,Overwhelmingly Positive,140668,98,35.0,[],
48601,1326470,Sons Of The Forest,Very Positive,128626,83,30.0,[],
14401,359320,Elite Dangerous,Mostly Positive,68600,76,30.0,[],
47793,427520,Factorio,Overwhelmingly Positive,134384,96,35.0,[],


In [17]:
# Doom Eternal
rec = recommend(df, 976730, cs_matrix)
rec[['app_id', 'title', 'rating', 'user_reviews', 'positive_ratio', 'price_final','tags', 'description']]

,app_id,title,rating,user_reviews,positive_ratio,price_final,tags,description
15097,782330,DOOM Eternal,Very Positive,153466,91,40.0,[],
47760,394360,Hearts of Iron IV,Very Positive,176243,92,40.0,[],
15719,1172620,Sea of Thieves 2023 Edition,Very Positive,253844,90,40.0,[],
12658,594650,Hunt: Showdown,Very Positive,136385,83,40.0,[],
15397,489830,The Elder Scrolls V: Skyrim Special Edition,Very Positive,138503,94,40.0,[],
15809,534380,Dying Light 2 Stay Human,Mostly Positive,113138,79,30.0,[],
12740,294100,RimWorld,Overwhelmingly Positive,140668,98,35.0,[],
48601,1326470,Sons Of The Forest,Very Positive,128626,83,30.0,[],
14401,359320,Elite Dangerous,Mostly Positive,68600,76,30.0,[],
47793,427520,Factorio,Overwhelmingly Positive,134384,96,35.0,[],


In [ ]:
# Geometry Dash
rec = recommend(df, 322170, cs_matrix)
rec[['app_id', 'title', 'rating', 'user_reviews', 'positive_ratio', 'price_final','tags', 'description']]

,app_id,title,rating,user_reviews,positive_ratio,price_final,tags,description
15077,1237970,Titanfall® 2,Very Positive,154419,94,3.0,[],
15281,1794680,Vampire Survivors,Overwhelmingly Positive,197109,98,5.0,[],
47791,431960,Wallpaper Engine,Overwhelmingly Positive,637341,98,4.0,[],
47817,774171,Muse Dash,Very Positive,86591,90,3.0,[],
14095,289070,Sid Meier’s Civilization® VI,Very Positive,201288,85,6.0,[],
50781,945360,Among Us,Very Positive,587821,92,3.0,[],
47394,3590,Plants vs. Zombies GOTY Edition,Overwhelmingly Positive,100864,97,5.0,[],
47790,433340,Slime Rancher,Overwhelmingly Positive,97997,97,5.0,[],
48013,674940,Stick Fight: The Game,Very Positive,89774,93,5.0,[],
47548,238320,Outlast,Overwhelmingly Positive,75391,96,3.0,[],
